# Comprobar el entorno de Colab para NOX (sin dataset)

Prepara y comprueba el entorno antes del entrenamiento de T05. **Solo comprueba la GPU y las herramientas.**

- No pide ni sube ningún ZIP.
- No entrena, no evalúa y no descarga pesos.
- No monta Drive ni escribe resultados.
- No crea `ONE_RUN.lock`, así que no consume la única ejecución de T05.

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU** (gratuita). Si Colab no asigna GPU, detenerse; no comprar unidades de cómputo.

Ejecuta las tres celdas en orden. Repetir esta comprobación no entrena; la instalación sí puede modificar paquetes.

**Ruta recomendada para una sola sesión:** abre directamente `notebooks/nox_text_training.ipynb` y ejecuta sus dos primeras celdas, que comprueban GPU, instalación y versión antes del dataset. Continúa dentro de ese mismo cuaderno solo si pasan.

Este cuaderno independiente es un diagnóstico alternativo: su éxito no garantiza que otro notebook tenga el mismo entorno. No abras otro cuaderno suponiendo que conserva la GPU o los paquetes. Si algo falla o se pide reiniciar, detenerse antes de entrenar; no comprar unidades.

In [ ]:
import json, platform, subprocess, sys
import torch
GPU_CHECK_PASSED = False
ENVIRONMENT_READY = False

if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU CUDA. Selecciona GPU gratuita y vuelve a comenzar.")
print("GPU:", torch.cuda.get_device_name(0), "| Python:", sys.version.split()[0], "| Torch:", torch.__version__)
probe = torch.ones((16, 16), device="cuda")
assert (probe @ probe)[0, 0].item() == 16
del probe
torch.cuda.empty_cache()
environment = {"python": sys.version, "platform": platform.platform(), "torch": torch.__version__,
               "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0)}
GPU_CHECK_PASSED = True


In [ ]:
import importlib.metadata, importlib.util
ENVIRONMENT_READY = False
if not globals().get("GPU_CHECK_PASSED", False) or not torch.cuda.is_available():
    raise RuntimeError("La comprobación GPU debe pasar y CUDA debe seguir disponible antes de instalar")
if importlib.util.find_spec("jedi") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "jedi==0.19.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "ultralytics==8.4.189"], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
from ultralytics import settings
settings.update({k: False for k in ("sync", "wandb", "mlflow", "comet", "clearml") if k in settings})
environment["ultralytics"] = importlib.metadata.version("ultralytics")
assert environment["ultralytics"] == "8.4.189", "Versión distinta de la fijada para el proyecto"
if not torch.cuda.is_available():
    raise RuntimeError("La GPU dejó de estar disponible; detenerse antes del entrenamiento")
ENVIRONMENT_READY = True


In [ ]:
if not globals().get("ENVIRONMENT_READY", False) or not torch.cuda.is_available():
    raise RuntimeError("El entorno no está listo; no continuar al entrenamiento")
assert environment["ultralytics"] == "8.4.189", "Versión distinta de la fijada para el proyecto"
print(json.dumps(environment, indent=2))
print("Entorno correcto. No se ha usado ningún dato ni se ha entrenado nada.")
print("Para una sola sesión, usa las dos primeras celdas de notebooks/nox_text_training.ipynb dentro de ese cuaderno. Este diagnóstico no comparte automáticamente su entorno.")